In [ ]:
import pandas as pd

df = pd.read_csv("/content/rows.csv", low_memory=False)

# Keep only the real text + label columns
df = df[["Consumer complaint narrative", "Product"]]
df.columns = ["text", "category"]

print("Total rows:", df.shape[0])
df.head()


Total rows: 81154


,text,category
0,NaN,Checking or savings account
1,NaN,Checking or savings account
2,NaN,Debt collection
3,NaN,"Credit reporting, credit repair services, or o..."
4,NaN,Checking or savings account


In [ ]:
df = df[df["text"].notna()]
df = df[df["text"].str.strip() != ""]

print("Rows with real text:", df.shape[0])


Rows with real text: 15222


In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

# Remove very short complaints (noise)
df = df[df["clean_text"].str.len() > 20]

print("Rows after cleaning:", df.shape[0])


Rows after cleaning: 15220


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label"] = le.fit_transform(df["category"])

print("Number of categories:", len(le.classes_))
print("Categories:", le.classes_)


Number of categories: 9
Categories: ['Checking or savings account' 'Credit card or prepaid card'
 'Credit reporting, credit repair services, or other personal consumer reports'
 'Debt collection' 'Money transfer, virtual currency, or money service'
 'Mortgage' 'Payday loan, title loan, or personal loan' 'Student loan'
 'Vehicle loan or lease']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])


Train size: 12176
Test size: 3044


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=2000,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF shape:", X_train_tfidf.shape)


TF-IDF shape: (12176, 2000)


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, n_jobs=-1)
model.fit(X_train_tfidf, y_train)

print("Model training completed")


Model training completed


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))


Accuracy: 81.93 %

Classification Report:

                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.78      0.81      0.79       195
                                                 Credit card or prepaid card       0.78      0.78      0.78       295
Credit reporting, credit repair services, or other personal consumer reports       0.84      0.90      0.87      1268
                                                             Debt collection       0.80      0.80      0.80       667
                          Money transfer, virtual currency, or money service       0.74      0.52      0.61        60
                                                                    Mortgage       0.87      0.90      0.89       261
                                   Payday loan, title loan, or personal loan       0.72      0.34      0.46        67
            

In [ ]:
def predict_category(text):
    text = clean_text(text)
    vec = tfidf.transform([text])
    pred = model.predict(vec)
    return le.inverse_transform(pred)[0]

sample = "There has been no response from the bank regarding my checking account for many days"
print("Sample:", sample)
print("Predicted Category:", predict_category(sample))
sample = "My credit card was charged twice for the same transaction and the bank is not responding"
print("Sample:", sample)
print("Predicted Category:", predict_category(sample))

sample = "I applied for a home loan but the mortgage process has been delayed for months without any update"
print("Sample:", sample)
print("Predicted Category:", predict_category(sample))
sample = "A debt collection agency keeps calling me for a loan I never took"
print("Sample:", sample)
print("Predicted Category:", predict_category(sample))




Sample: There has been no response from the bank regarding my checking account for many days
Predicted Category: Checking or savings account
Sample: My credit card was charged twice for the same transaction and the bank is not responding
Predicted Category: Credit card or prepaid card
Sample: I applied for a home loan but the mortgage process has been delayed for months without any update
Predicted Category: Mortgage
Sample: A debt collection agency keeps calling me for a loan I never took
Predicted Category: Debt collection


In [ ]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
